# Trend Analysis and Delta Lake Time Travel

The reconciliation notebooks (NB01 through NB05) each produce a **point-in-time
snapshot**: how many discrepancies exist today, what's the dollar exposure right
now. That's useful for daily operations but **doesn't answer** the questions that
come up in periodic reviews and **root cause investigations**.

As examples:
1. Are overcharges increasing over time? Did something change last month that 
caused a spike?
2. When we investigate a specific member's overcharge, what did the accumulator 
table actually look like at the moment the claim was adjudicated?

This notebook covers two things:

| Part | What it does |
|------|-------------|
| A - Daily Reconciliation Trend | Simulates 90 days of NB03 output, builds a trend with weekly rollups, and runs spike detection with seasonal decomposition to filter out weekend noise |
| B - Delta Lake Time Travel | Creates a versioned accumulator table to demonstrate forensic investigation of a specific member's balance at the exact moment a claim was adjudicated |

Both capabilities rely on Delta Lake's versioned storage - we don't need
separate audit tables or snapshot copies because the history is built into the
table format itself.

In [0]:
%run ./NB_00_pipeline_utilities

# NB00 - Pipeline Utilities
**Eligibility & Accumulator**

In production, a reconciliation pipeline has dozens of checks spread across multiple notebooks.
If one those check fails, say the TPA eligibility table got dropped by an upstream job, you don't
want the entire pipeline to crash and leave you with nothing. You still want the accumulator
reconciliation, the copay audit, and the rollover checks to run and produce findings.

You also need to know exactly what ran, what passed, and what eventually broke.

This notebook provides shared utilities that every downstream notebook imports via
`%run ./NB_00_pipeline_utilities`. It handles three things:

| Utility | What it does |
|---------|-------------|
| Structured logging | JSON audit entries to `pbm_sim.pipeline_audit_log`, queryable by run, notebook, section |
| Error handling decorator | Wraps each check in try/except so one failure doesn't kill the entire pipeline |
| Table validation helpers | Pre-flight checks before a notebook starts its logic (do the table exists? does it has rows? does it have right columns?) |

These functions are used in NB01 through NB07.

## Structured Logging

Every pipeline run gets a unique `run_id` based on the UTC timestamp when the run started.
It ties together all the log entries from a single execution across notebooks.
If the orchestrator kicks off NB01 through NB07 sequentially, they all share the same run_id.

Log entries go to two places:
1. **stdout** as `[PIPELINE_LOG] {...}` - captured by Databricks job logs, useful for real-time monitoring
2. **Delta table** `pbm_sim.pipeline_audit_log` - the permanent audit trail you can query later

Each entry records the notebook, section, outcome (SUCCESS/WARNING/ERROR/SKIPPED),
row count, timing, and a free-text detail field. The detail field is truncated at 2,000
characters to prevent a full stack trace from bloating the table.

## Error Handling Decorator

This is the core safety mechanism. In a reconciliation pipeline, each notebook runs
multiple independent checks. Take NB03 (accumulator reconciliation) - it checks for
deductible overcharges AND OOPM overcharges. If the deductible check throws an exception
(maybe a column was renamed upstream), you still want the OOPM check to run and produce
its findings. Without this decorator, an unhandled exception in the deductible check
would kill the notebook, and you'd lose the OOPM results too.

The decorator does three things on failure:
1. Catches the exception and logs it as ERROR with the full traceback
2. Returns an empty DataFrame with the expected schema so downstream unions don't break
3. Lets the notebook continue to the next section

On success, it logs the section as SUCCESS with the row count and wall-clock time.

**Usage in a downstream notebook:**
```python
@pipeline_section("NB_03", "deductible_overcharge", finding_schema=convergence_schema)
def run_deductible_check():
    # ... compute df ...
    return df

df_result = run_deductible_check()
# If the check fails, df_result is an empty DataFrame with the right columns
# so the union at the end of the notebook still works
```

## Table Validation Helpers (Pre-flight Checks)

Before a notebook starts its actual logic, it should verify that the tables it depends on
exist and have data. Without this, a missing table produces a cryptic *AnalysisException*
50 cells into the notebook, and you spend 10 minutes figuring out that the problem was
simply that NB00 didn't run.

These are the pre-flight checks:
- **The Table exists?** Catches the case where the upstream notebook didn't run or the table got dropped
- **Are the table empty?** The table exists but has zero rows - maybe the data generator ran but the filter
  produced nothing. This is different from "table doesn't exist" and needs a different response
- **Are the required columns present?** Schema drift - a column was renamed or removed upstream. Better to catch
  this up front with a clear message than to get a "Column 'xyz' not found" error mid-analysis

## Pipeline Metadata

Convenience function for embedding run metadata into output tables.
When NB07 unions all findings, each row can carry the run_id so you can
trace any finding back to the exact pipeline execution that produced it.

Pipeline utilities loaded. Run ID: RUN_20261019_115753


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from datetime import date, timedelta
import random

random.seed(99)
spark.sql("USE pbm_sim")

NB = "NB_06"

log_entry(NB, "load_source_tables", "SUCCESS")


[PIPELINE_LOG] {"run_id": "RUN_20261019_115753", "timestamp": "2026-10-19T11:57:54.824698+00:00", "notebook": "NB_06", "section": "load_source_tables", "status": "SUCCESS", "row_count": 0, "detail": "", "duration_sec": 0.0}


## Part A - Daily Reconciliation Trend

In production, the accumulator reconciliation (NB03) would run every morning
after the TPA file lands. Each run produces a count of overcharged members
and a total dollar exposure. Logging those numbers daily lets us track
whether the situation is stable, improving, or deteriorating.

Since the synthetic data is a single snapshot, we simulate 90 days of
reconciliation results to build a realistic trend. The simulation models
the patterns you'd see in a real environment: gradual increase through the
benefit year as more members approach their limits, occasional spikes from
TPA file delays, and a drop after a known remediation event.

In [0]:
# Simulate 90 days of daily reconciliation results (July through September 2026).
# This models what the output of running NB03 daily would look like if we
# logged the results each time.

recon_log = []
base_date = date(2026, 7, 1)

# Baseline parameters that shift over the 90-day window
base_ded_overcharges = 18     # Starting count in early July
base_oopm_overcharges = 6
base_ded_dollars = 5200.0
base_oopm_dollars = 1800.0

for day_offset in range(90):
    run_date = base_date + timedelta(days=day_offset)

    # Gradual upward trend as more members accumulate spending through the year
    trend_factor = 1.0 + (day_offset / 90) * 0.6

    # Daily noise
    noise = random.uniform(0.85, 1.15)

    # Simulate a TPA file delay spike on days 35-38 (early August).
    # When a TPA misses a file, the PBM adjudicates with data that's 2 days stale
    # instead of 1, roughly doubling the overcharge count.
    spike = 1.0
    if 35 <= day_offset <= 38:
        spike = 2.2

    # Simulate a remediation event on day 55 (late August).
    # The team identified a parsing issue in TPA 2's file and fixed it,
    # reducing overcharges by ~30%.
    remediation = 1.0
    if day_offset >= 55:
        remediation = 0.70

    ded_count = int(base_ded_overcharges * trend_factor * noise * spike * remediation)
    oopm_count = int(base_oopm_overcharges * trend_factor * noise * spike * remediation)
    ded_dollars = round(base_ded_dollars * trend_factor * noise * spike * remediation, 2)
    oopm_dollars = round(base_oopm_dollars * trend_factor * noise * spike * remediation, 2)

    # Weekends have lower claim volume, so fewer new overcharges surface
    if run_date.weekday() >= 5:
        ded_count = int(ded_count * 0.4)
        oopm_count = int(oopm_count * 0.3)
        ded_dollars = round(ded_dollars * 0.4, 2)
        oopm_dollars = round(oopm_dollars * 0.3, 2)

    total_reconciled = random.randint(1780, 1860)

    recon_log.append((
        run_date,
        total_reconciled,
        ded_count, oopm_count, ded_count + oopm_count,
        ded_dollars, oopm_dollars, round(ded_dollars + oopm_dollars, 2),
        round((ded_count + oopm_count) / total_reconciled * 100, 2)
    ))

recon_schema = [
    "run_date", "members_reconciled",
    "ded_overcharges", "oopm_overcharges", "total_overcharges",
    "ded_exposure", "oopm_exposure", "total_exposure",
    "overcharge_rate_pct"
]

df_recon_log = spark.createDataFrame(recon_log, recon_schema)

(df_recon_log.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("pbm_sim.daily_recon_log"))

print(f"Daily reconciliation log: {df_recon_log.count()} days")
display(df_recon_log.orderBy("run_date"))

log_entry(NB, "daily_recon_log_created", "SUCCESS",
          detail=f"days_simulated=90, table=pbm_sim.daily_recon_log")

Daily reconciliation log: 90 days


run_date,members_reconciled,ded_overcharges,oopm_overcharges,total_overcharges,ded_exposure,oopm_exposure,total_exposure,overcharge_rate_pct
2026-07-01,1848,17,5,22,5017.59,1736.86,6754.45,1.19
2026-07-02,1859,15,5,20,4590.55,1589.04,6179.59,1.08
2026-07-03,1833,18,6,24,5252.38,1818.13,7070.51,1.31
2026-07-04,1855,7,1,8,2263.04,587.52,2850.56,0.43
2026-07-05,1827,6,1,7,1953.97,507.28,2461.25,0.38
2026-07-06,1807,20,6,26,5892.92,2039.86,7932.78,1.44
2026-07-07,1802,16,5,21,4843.85,1676.72,6520.57,1.17
2026-07-08,1823,20,6,26,6014.09,2081.8,8095.89,1.43
2026-07-09,1847,16,5,21,4792.06,1658.79,6450.85,1.14
2026-07-10,1791,20,6,26,5805.31,2009.53,7814.84,1.45


[PIPELINE_LOG] {"run_id": "RUN_20261019_115753", "timestamp": "2026-10-19T11:57:59.948363+00:00", "notebook": "NB_06", "section": "daily_recon_log_created", "status": "SUCCESS", "row_count": 0, "detail": "days_simulated=90, table=pbm_sim.daily_recon_log", "duration_sec": 0.0}


### Trend Visualization

Use the chart icon on the output below to switch to a line chart.
Set the X axis to `run_date` and the Y axis to `total_exposure`.

The three phases should be visible: gradual climb through July, the spike
in early August when TPA 2 missed file deliveries, and the drop in late
August after the parsing fix was deployed.

In [0]:
# Weekly aggregation for a cleaner trend view.
# Daily numbers are noisy - rolling up to week-level makes
# the underlying pattern easier to spot in a chart.
df_weekly = (
    df_recon_log
    .withColumn("week_start", F.date_trunc("week", F.col("run_date")))
    .groupBy("week_start")
    .agg(
        F.round(F.avg("total_overcharges"), 1).alias("avg_daily_overcharges"),
        F.round(F.avg("total_exposure"), 2).alias("avg_daily_exposure"),
        F.round(F.avg("overcharge_rate_pct"), 2).alias("avg_overcharge_rate"),
        F.max("total_overcharges").alias("peak_day_overcharges"),
        F.round(F.max("total_exposure"), 2).alias("peak_day_exposure"),
    )
    .orderBy("week_start")
)

print("Weekly reconciliation trend:")
display(df_weekly)

Weekly reconciliation trend:


week_start,avg_daily_overcharges,avg_daily_exposure,avg_overcharge_rate,peak_day_overcharges,peak_day_exposure
2026-06-29T00:00:00.000Z,16.2,5063.27,0.88,24,7070.51
2026-07-06T00:00:00.000Z,19.3,6001.42,1.06,26,8095.89
2026-07-13T00:00:00.000Z,19.9,6119.51,1.09,29,8624.56
2026-07-20T00:00:00.000Z,21.1,6528.47,1.17,30,9042.72
2026-07-27T00:00:00.000Z,22.9,6943.58,1.27,32,9355.66
2026-08-03T00:00:00.000Z,41.9,12580.81,2.3,70,20694.01
2026-08-10T00:00:00.000Z,25.9,7817.83,1.43,34,10173.05
2026-08-17T00:00:00.000Z,24.9,7590.29,1.37,32,9477.54
2026-08-24T00:00:00.000Z,19.6,5946.3,1.08,34,10153.06
2026-08-31T00:00:00.000Z,19.6,6036.32,1.08,26,7926.15


### Spike Detection with Seasonal Decomposition

A naive rolling average flags Monday rebounds as spikes because weekends
have ~40% lower claim volume. To avoid false alarms, we decompose the
signal into three components:

1. Day-of-week seasonal factor - average ratio of each weekday's
   overcharge count to the overall mean. Weekends get factors below 1.0,
   weekdays above 1.0.
2. Seasonally adjusted series - raw count divided by the DOW factor.
   This normalizes weekends up and weekdays down so the rolling window
   operates on a level baseline.
3. Spike flag - applied to the adjusted series using a 14-day
   rolling average +/- 2 standard deviations. Because the seasonal
   pattern has been removed, only genuine operational events (TPA file
   delays, configuration changes) trigger alerts.

In [0]:
# Step 1: Compute day-of-week seasonal factors.
# Without this, every Monday looks like a spike because Sunday's
# count was artificially low.
df_with_dow = df_recon_log.withColumn("day_of_week", F.dayofweek("run_date"))

overall_mean = df_recon_log.agg(F.avg("total_overcharges")).collect()[0][0]

df_dow_factors = (
    df_with_dow
    .groupBy("day_of_week")
    .agg(F.avg("total_overcharges").alias("dow_avg"))
    .withColumn("seasonal_factor", F.round(F.col("dow_avg") / F.lit(overall_mean), 4))
)

print("Day-of-week seasonal factors (1=Sun, 7=Sat):")
display(df_dow_factors.orderBy("day_of_week"))

Day-of-week seasonal factors (1=Sun, 7=Sat):


day_of_week,dow_avg,seasonal_factor
1,8.846153846153847,0.393
2,26.923076923076923,1.196
3,25.083333333333332,1.1143
4,28.846153846153847,1.2814
5,29.076923076923077,1.2917
6,29.23076923076923,1.2985
7,9.76923076923077,0.434


In [0]:
# Step 2: Seasonally adjust the overcharge series
df_adjusted = (
    df_with_dow
    .join(df_dow_factors.select("day_of_week", "seasonal_factor"), on="day_of_week")
    .withColumn(
        "adjusted_overcharges",
        F.round(F.col("total_overcharges") / F.col("seasonal_factor"), 1),
    )
)

# Step 3: Spike detection on the adjusted series.
# 14-day window gives enough history to establish a stable baseline
# without being so long that it absorbs the spike into the average.
window_14d = (
    Window
    .orderBy("run_date")
    .rowsBetween(-13, 0)
)

df_spike = (
    df_adjusted
    .withColumn("rolling_avg_14d", F.round(F.avg("adjusted_overcharges").over(window_14d), 1))
    .withColumn("rolling_std_14d", F.round(F.stddev("adjusted_overcharges").over(window_14d), 1))
    .withColumn(
        "upper_threshold",
        F.round(F.col("rolling_avg_14d") + 2 * F.col("rolling_std_14d"), 1),
    )
    .withColumn(
        "is_spike",
        F.col("adjusted_overcharges") > F.col("upper_threshold"),
    )
    .withColumn(
        "spike_magnitude",
        F.when(
            F.col("is_spike"),
            F.round(
                (F.col("adjusted_overcharges") - F.col("rolling_avg_14d")) / F.col("rolling_std_14d"),
                2,
            ),
        ),
    )
)

spike_days = df_spike.filter(F.col("is_spike"))
spike_count = spike_days.count()

print(f"Spike days detected (seasonally adjusted, >2 sigma above 14-day rolling avg): {spike_count}")

if spike_count > 0:
    display(
        spike_days
        .select(
            "run_date", "day_of_week", "total_overcharges",
            "seasonal_factor", "adjusted_overcharges",
            "rolling_avg_14d", "upper_threshold", "spike_magnitude",
            "total_exposure",
        )
        .orderBy("run_date")
    )

log_entry(NB, "spike_detection", "SUCCESS",
          detail=f"spike_days={spike_count}, method=seasonal_decomposition_2sigma")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Spike days detected (seasonally adjusted, >2 sigma above 14-day rolling avg): 5


run_date,day_of_week,total_overcharges,seasonal_factor,adjusted_overcharges,rolling_avg_14d,upper_threshold,spike_magnitude,total_exposure
2026-07-19,1,10,0.393,25.4,19.7,24.7,2.28,3332.68
2026-08-05,4,69,1.2814,53.8,24.7,42.9,3.2,20352.62
2026-08-06,5,70,1.2917,54.2,27.3,50.7,2.3,20694.01
2026-09-14,2,28,1.196,23.4,19.1,23.3,2.05,8204.39
2026-09-27,1,10,0.393,25.4,20.7,25.1,2.14,3126.95


[PIPELINE_LOG] {"run_id": "RUN_20261019_115753", "timestamp": "2026-10-19T11:58:06.191111+00:00", "notebook": "NB_06", "section": "spike_detection", "status": "SUCCESS", "row_count": 0, "detail": "spike_days=5, method=seasonal_decomposition_2sigma", "duration_sec": 0.0}


## Part B - Delta Lake Time Travel

Delta Lake tracks every write to a table as a versioned transaction. When
we need to investigate a specific overcharge, we can ask: what did the
accumulator table look like at version N, which was the version in effect
when the claim was adjudicated?

To demonstrate this, we create a versioned accumulator table and write
several updates to it, simulating the nightly TPA file loads. Then we
query historical versions to trace how a member's balance changed over time.

In [0]:
# Build a small versioned table simulating 4 nightly TPA loads for a handful
# of members. Each write is a separate version in Delta's transaction log.

sample_members = ["MBR1000010", "MBR1000025", "MBR1000042", "MBR1000058", "MBR1000071"]

# Version 0: Initial load (Sep 10 file)
v0_data = [
    ("MBR1000010", "PLN002", 2026, 1200.00, 1800.00, date(2026, 9, 10)),
    ("MBR1000025", "PLN002", 2026,  800.00, 1100.00, date(2026, 9, 10)),
    ("MBR1000042", "PLN004", 2026, 2900.00, 4200.00, date(2026, 9, 10)),
    ("MBR1000058", "PLN004", 2026, 1500.00, 2000.00, date(2026, 9, 10)),
    ("MBR1000071", "PLN001", 2026,  400.00,  600.00, date(2026, 9, 10)),
]

v0_schema = ["member_id", "plan_id", "benefit_year",
             "medical_deductible_applied", "medical_oopm_applied", "file_date"]

df_v0 = spark.createDataFrame(v0_data, v0_schema)
(df_v0.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("pbm_sim.tpa_accum_versioned"))

print("Version 0 written (Sep 10 file)")

Version 0 written (Sep 10 file)


In [0]:
# Version 1: Sep 11 file. MBR1000042 had a large medical claim, balance jumped.
v1_data = [
    ("MBR1000010", "PLN002", 2026, 1200.00, 1800.00, date(2026, 9, 11)),
    ("MBR1000025", "PLN002", 2026,  850.00, 1200.00, date(2026, 9, 11)),
    ("MBR1000042", "PLN004", 2026, 3500.00, 5800.00, date(2026, 9, 11)),  # Big jump
    ("MBR1000058", "PLN004", 2026, 1500.00, 2000.00, date(2026, 9, 11)),
    ("MBR1000071", "PLN001", 2026,  400.00,  600.00, date(2026, 9, 11)),
]

df_v1 = spark.createDataFrame(v1_data, v0_schema)
(df_v1.write.format("delta").mode("overwrite").saveAsTable("pbm_sim.tpa_accum_versioned"))
print("Version 1 written (Sep 11 file)")

Version 1 written (Sep 11 file)


In [0]:
# Version 2: Sep 12 file. MBR1000042 met the deductible, now accumulating toward OOPM.
v2_data = [
    ("MBR1000010", "PLN002", 2026, 1250.00, 1900.00, date(2026, 9, 12)),
    ("MBR1000025", "PLN002", 2026,  900.00, 1300.00, date(2026, 9, 12)),
    ("MBR1000042", "PLN004", 2026, 3500.00, 7200.00, date(2026, 9, 12)),  # OOPM climbing
    ("MBR1000058", "PLN004", 2026, 1650.00, 2200.00, date(2026, 9, 12)),
    ("MBR1000071", "PLN001", 2026,  450.00,  700.00, date(2026, 9, 12)),
]

df_v2 = spark.createDataFrame(v2_data, v0_schema)
(df_v2.write.format("delta").mode("overwrite").saveAsTable("pbm_sim.tpa_accum_versioned"))
print("Version 2 written (Sep 12 file)")

Version 2 written (Sep 12 file)


In [0]:
# Version 3: Sep 13 file. MBR1000042 is now past the OOPM limit.
v3_data = [
    ("MBR1000010", "PLN002", 2026, 1300.00, 2000.00, date(2026, 9, 13)),
    ("MBR1000025", "PLN002", 2026,  950.00, 1400.00, date(2026, 9, 13)),
    ("MBR1000042", "PLN004", 2026, 3500.00, 9200.00, date(2026, 9, 13)),  # Past OOPM (limit is 9000)
    ("MBR1000058", "PLN004", 2026, 1700.00, 2400.00, date(2026, 9, 13)),
    ("MBR1000071", "PLN001", 2026,  500.00,  800.00, date(2026, 9, 13)),
]

df_v3 = spark.createDataFrame(v3_data, v0_schema)
(df_v3.write.format("delta").mode("overwrite").saveAsTable("pbm_sim.tpa_accum_versioned"))
print("Version 3 written (Sep 13 file)")

log_entry(NB, "versioned_table_created", "SUCCESS",
          detail="4 versions written to pbm_sim.tpa_accum_versioned (Sep 10-13)")


Version 3 written (Sep 13 file)
[PIPELINE_LOG] {"run_id": "RUN_20261019_115753", "timestamp": "2026-10-19T11:58:18.842212+00:00", "notebook": "NB_06", "section": "versioned_table_created", "status": "SUCCESS", "row_count": 0, "detail": "4 versions written to pbm_sim.tpa_accum_versioned (Sep 10-13)", "duration_sec": 0.0}


### Table History

Delta Lake records every transaction. `DESCRIBE HISTORY` shows when each
version was written, which is the foundation for time travel queries.

In [0]:
%sql
DESCRIBE HISTORY pbm_sim.tpa_accum_versioned

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
3,2026-10-19T11:58:19.000Z,78178792503483,alessandroodesousa@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.enableDeletionVectors"":""true"",""databricks.internal.autoUpgrades.delta.checkpointPolicy"":""v2"",""delta.parquet.format.version"":""2.12.0"",""delta.checkpointPolicy"":""v2"",""io.unitycatalog.tableId"":""9a31acd0-2652-4050-a17a-a122b87fa78c""}, statsOnLoad -> true)",null,List(2743853735846218),fca5e24e-f1b1-460c-a2cd-85fb83fca435,1019-112444-mtoi6ejg-v2n,2,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 2239, numDeletionVectorsRemoved -> 0, numOutputRows -> 5, numOutputBytes -> 2238)",null,Databricks-Runtime/19.9.x-aarch64-photon-scala2.13
2,2026-10-19T11:58:17.000Z,78178792503483,alessandroodesousa@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.enableDeletionVectors"":""true"",""databricks.internal.autoUpgrades.delta.checkpointPolicy"":""v2"",""delta.parquet.format.version"":""2.12.0"",""delta.checkpointPolicy"":""v2"",""io.unitycatalog.tableId"":""9a31acd0-2652-4050-a17a-a122b87fa78c""}, statsOnLoad -> true)",null,List(2743853735846218),74c019df-cefd-4d17-bdea-919a0f37e6cb,1019-112444-mtoi6ejg-v2n,1,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 2239, numDeletionVectorsRemoved -> 0, numOutputRows -> 5, numOutputBytes -> 2239)",null,Databricks-Runtime/19.9.x-aarch64-photon-scala2.13
1,2026-10-19T11:58:15.000Z,78178792503483,alessandroodesousa@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.enableDeletionVectors"":""true"",""databricks.internal.autoUpgrades.delta.checkpointPolicy"":""v2"",""delta.parquet.format.version"":""2.12.0"",""delta.checkpointPolicy"":""v2"",""io.unitycatalog.tableId"":""9a31acd0-2652-4050-a17a-a122b87fa78c""}, statsOnLoad -> true)",null,List(2743853735846218),b8520b37-e294-4702-bba1-b547526a12e7,1019-112444-mtoi6ejg-v2n,0,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 2238, numDeletionVectorsRemoved -> 0, numOutputRows -> 5, numOutputBytes -> 2239)",null,Databricks-Runtime/19.9.x-aarch64-photon-scala2.13
0,2026-10-19T11:58:13.000Z,78178792503483,alessandroodesousa@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.enableDeletionVectors"":""true"",""databricks.internal.autoUpgrades.delta.checkpointPolicy"":""v2"",""delta.parquet.format.version"":""2.12.0"",""delta.checkpointPolicy"":""v2"",""io.unitycatalog.tableId"":""9a31acd0-2652-4050-a17a-a122b87fa78c""}, statsOnLoad -> true)",null,List(2743853735846218),91af1efc-0447-4c88-bcaa-3493b158c6a0,1019-112444-mtoi6ejg-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 5, numOutputBytes -> 2238)",null,Databricks-Runtime/19.9.x-aarch64-photon-scala2.13


### Querying Historical Versions

Suppose a claim for MBR1000042 was adjudicated on Sep 11. At that point,
the PBM's most recent TPA file was the Sep 10 load (version 0). The PBM
didn't know about the large medical claim that showed up in the Sep 11
file (version 1).

We can query version 0 to see exactly what the PBM was working with
when it made its adjudication decision.

In [0]:
# What the PBM saw when it adjudicated the Sep 11 claim (version 0, the Sep 10 file).
# This is the data the adjudication engine used - stale by one day.
print("Version 0 (Sep 10 file) - what the PBM had at adjudication time:")
df_at_adjudication = (
    spark.read
    .format("delta")
    .option("versionAsOf", 0)
    .table("pbm_sim.tpa_accum_versioned")
    .filter(F.col("member_id") == "MBR1000042")
)
display(df_at_adjudication)


Version 0 (Sep 10 file) - what the PBM had at adjudication time:


member_id,plan_id,benefit_year,medical_deductible_applied,medical_oopm_applied,file_date
MBR1000042,PLN004,2026,2900.0,4200.0,2026-09-10


In [0]:
# What the actual balance was by the end of that day (version 1, the Sep 11 file).
# This is what the PBM should have known but didn't.
print("Version 1 (Sep 11 file) - actual balance after the medical claim processed:")
df_actual = (
    spark.read
    .format("delta")
    .option("versionAsOf", 1)
    .table("pbm_sim.tpa_accum_versioned")
    .filter(F.col("member_id") == "MBR1000042")
)
display(df_actual)

Version 1 (Sep 11 file) - actual balance after the medical claim processed:


member_id,plan_id,benefit_year,medical_deductible_applied,medical_oopm_applied,file_date
MBR1000042,PLN004,2026,3500.0,5800.0,2026-09-11


In [0]:
# The gap between what the PBM knew and reality, side by side.
# This is the core of a forensic investigation - showing exactly
# where the data lag caused the wrong adjudication.
print("MBR1000042 on PLN004 (individual deductible limit: $3,500, OOPM limit: $9,000)")
print("")
print("At adjudication (Sep 10 data):")
row_v0 = df_at_adjudication.collect()[0]
print(f"  Medical deductible applied: ${row_v0['medical_deductible_applied']:,.2f}")
print(f"  Medical OOPM applied:       ${row_v0['medical_oopm_applied']:,.2f}")
print("")

print("Actual balance (Sep 11 data):")
row_v1 = df_actual.collect()[0]
print(f"  Medical deductible applied: ${row_v1['medical_deductible_applied']:,.2f}")
print(f"  Medical OOPM applied:       ${row_v1['medical_oopm_applied']:,.2f}")
print("")

ded_gap = row_v1['medical_deductible_applied'] - row_v0['medical_deductible_applied']
oopm_gap = row_v1['medical_oopm_applied'] - row_v0['medical_oopm_applied']
print(f"Gap the PBM didn't know about:")
print(f"  Deductible: ${ded_gap:,.2f}")
print(f"  OOPM:       ${oopm_gap:,.2f}")
print(f"\nAny pharmacy claim adjudicated on Sep 11 for this member used the")
print(f"Sep 10 balances. If the pharmacy-side spending pushed the combined")
print(f"total over the plan limit, the overcharge happened in this gap window.")

log_entry(NB, "time_travel_investigation", "SUCCESS",
          detail=f"member=MBR1000042, ded_gap={ded_gap:.2f}, oopm_gap={oopm_gap:.2f}")


MBR1000042 on PLN004 (individual deductible limit: $3,500, OOPM limit: $9,000)

At adjudication (Sep 10 data):
  Medical deductible applied: $2,900.00
  Medical OOPM applied:       $4,200.00

Actual balance (Sep 11 data):
  Medical deductible applied: $3,500.00
  Medical OOPM applied:       $5,800.00

Gap the PBM didn't know about:
  Deductible: $600.00
  OOPM:       $1,600.00

Any pharmacy claim adjudicated on Sep 11 for this member used the
Sep 10 balances. If the pharmacy-side spending pushed the combined
total over the plan limit, the overcharge happened in this gap window.
[PIPELINE_LOG] {"run_id": "RUN_20261019_115753", "timestamp": "2026-10-19T11:58:28.842548+00:00", "notebook": "NB_06", "section": "time_travel_investigation", "status": "SUCCESS", "row_count": 0, "detail": "member=MBR1000042, ded_gap=600.00, oopm_gap=1600.00", "duration_sec": 0.0}


### Member Balance Progression

Querying all four versions lets us see the full trajectory of a member's
medical accumulator over the file delivery period. This is the kind of
view an analyst pulls when investigating a member complaint or an
overcharge flagged in the daily reconciliation.

In [0]:
# Build a progression view across all 4 versions for MBR1000042.
# Each version represents one nightly TPA file load, so this
# shows how the balance evolved day by day.
progression_rows = []
for v in range(4):
    df_ver = (
        spark.read
        .format("delta")
        .option("versionAsOf", v)
        .table("pbm_sim.tpa_accum_versioned")
        .filter(F.col("member_id") == "MBR1000042")
        .withColumn("table_version", F.lit(v))
    )
    progression_rows.append(df_ver)

from functools import reduce
df_progression = reduce(lambda a, b: a.union(b), progression_rows)

print("MBR1000042 accumulator progression across 4 nightly TPA loads:")
display(
    df_progression
    .select(
        "table_version", "file_date",
        "medical_deductible_applied", "medical_oopm_applied"
    )
    .orderBy("table_version")
)

log_entry(NB, "balance_progression", "SUCCESS",
          detail="member=MBR1000042, versions=0-3")

MBR1000042 accumulator progression across 4 nightly TPA loads:


table_version,file_date,medical_deductible_applied,medical_oopm_applied
0,2026-09-10,2900.0,4200.0
1,2026-09-11,3500.0,5800.0
2,2026-09-12,3500.0,7200.0
3,2026-09-13,3500.0,9200.0


[PIPELINE_LOG] {"run_id": "RUN_20261019_115753", "timestamp": "2026-10-19T11:58:32.933240+00:00", "notebook": "NB_06", "section": "balance_progression", "status": "SUCCESS", "row_count": 0, "detail": "member=MBR1000042, versions=0-3", "duration_sec": 0.0}


## Summary

In [0]:
print(f"{'=' * 60}")
print(f"  TREND ANALYSIS & TIME TRAVEL RESULTS")
print(f"{'=' * 60}")
print(f"  Daily recon log:          90 days (Jul-Sep 2026)")
print(f"  Spike days detected:      {spike_count}")
print(f"{'=' * 60}")
print(f"  Time travel demo table:   pbm_sim.tpa_accum_versioned")
print(f"  Versions written:         4 (Sep 10-13)")
print(f"  Investigation member:     MBR1000042")
print(f"    Deductible gap:         ${ded_gap:,.2f}")
print(f"    OOPM gap:               ${oopm_gap:,.2f}")
print(f"{'=' * 60}")

  TREND ANALYSIS & TIME TRAVEL RESULTS
  Daily recon log:          90 days (Jul-Sep 2026)
  Spike days detected:      5
  Time travel demo table:   pbm_sim.tpa_accum_versioned
  Versions written:         4 (Sep 10-13)
  Investigation member:     MBR1000042
    Deductible gap:         $600.00
    OOPM gap:               $1,600.00


## Investigation Notes

The trend analysis and time travel capability serve different audiences.

The daily reconciliation log and spike detection feed into management
reporting: is the accumulator sync working well overall, and are things
getting better or worse? The seasonal decomposition removes day-of-week
effects so that weekend-to-Monday volume swings don't create false
alerts. Only genuine operational events (TPA file delays, configuration
changes, remediation deploys) trigger spike flags.

The time travel queries feed into individual case investigation: for this
specific member and this specific claim, what went wrong and when?

Both rely on the same underlying infrastructure. Delta Lake's versioned
storage means we don't need to maintain separate audit tables or snapshot
copies. The history is built into the table format itself. That's one of
the practical advantages of running this workload on Databricks rather
than a traditional data warehouse where you'd need to build the
versioning layer yourself.

**Cleanup note:** The `tpa_accum_versioned` table was created here for
demonstration purposes with hardcoded member IDs from the synthetic data.
In production, time travel works against the actual `tpa_accumulator_feed`
table since every nightly file load creates a new version automatically
when using Delta Lake.

In [0]:
log_summary(NB)


  PIPELINE LOG SUMMARY: NB_06 (run: RUN_20261019_115753)
  [+] load_source_tables: SUCCESS
  [+] daily_recon_log_created: SUCCESS
  [+] spike_detection: SUCCESS
  [+] versioned_table_created: SUCCESS
  [+] time_travel_investigation: SUCCESS
  [+] balance_progression: SUCCESS

